# 2 · Describing an attack by what it does

The effect-level scenario library and its episodes. See
[Lesson 2](../docs/lessons/02-describing-an-attack-by-what-it-does.md).

In [1]:
import numpy as np

from cyberrel.attack_scenario_library import load_scenarios

lib = load_scenarios()
for s in lib.values():
    print(f"{s.name:22s} {s.kind.value:9s} -> {s.target_group:10s} "
          f"rate {s.rate_per_hour:.0e}/h  response {s.response_time:.2f} h  {s.technique or '-'}")

bus_flood_dos          denial    -> data_bus   rate 5e-04/h  response 1.25 h  T0814
gnss_spoofing          integrity -> nav_source rate 2e-04/h  response 2.50 h  -
false_nav_injection    integrity -> data_bus   rate 1e-04/h  response 3.00 h  T0856
message_delay_replay   delay     -> data_bus   rate 3e-04/h  response 2.00 h  T0830


Sampled episode lengths against the exact $E[\min(D, c)]$:

In [2]:
from cyberrel.attack_to_metric_translation import episode_means

rng = np.random.default_rng(1)
for s in lib.values():
    deny, length = s.sample_episodes(rng, 100_000)
    l_deny, l_degrade = episode_means(s)
    sampled = length[deny].mean() if deny.any() else 0.0
    print(f"{s.name:22s} P(deny) {deny.mean():.3f}  mean deny {sampled:.3f} h  exact {l_deny:.3f} h")

bus_flood_dos          P(deny) 1.000  mean deny 0.930 h  exact 0.929 h
gnss_spoofing          P(deny) 0.099  mean deny 2.500 h  exact 2.500 h
false_nav_injection    P(deny) 0.500  mean deny 3.000 h  exact 3.000 h
message_delay_replay   P(deny) 1.000  mean deny 1.458 h  exact 1.460 h


A delay below the deadline only degrades:

In [3]:
slow = lib["message_delay_replay"].with_changes(added_latency_ms=45.0)
print(slow.exceeds_deadline, slow.deny_probability)

False 0.0
